In [2]:
from pathlib import Path
import pandas as pd

In [3]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRANSCRIPTS_PATH = REPO_ROOT / "data" / "processed" / "transcripts.csv"

assert TRANSCRIPTS_PATH.exists(), f"Missing: {TRANSCRIPTS_PATH}"

df = pd.read_csv(TRANSCRIPTS_PATH, parse_dates = ["date_parsed"])

# Day 1 retro safety asserts
assert df["ticker"].nunique() == 30, f"Expected 30 tickers, got {df['ticker'.nuinque()]}"
assert len(df) == 274, f"Expected 274 transcripts, got {len(df)}"

print(f"Shape: {df.shape}")
print(f"Columns + dtypes:\n{df.dtypes}")
print(f"Date range: {df['date_parsed'].min().date()} -> {df['date_parsed'].max().date()}")

Shape: (274, 5)
Columns + dtypes:
ticker                    str
quarter                   str
date_parsed    datetime64[us]
date_raw                  str
transcript                str
dtype: object
Date range: 2019-06-25 -> 2023-02-02


In [4]:
df["transcript_char_count"] = df["transcript"].str.len()
df["transcript_word_count"] = df["transcript"].str.split().str.len()

df[["transcript_char_count", "transcript_word_count"]].describe()

,transcript_char_count,transcript_word_count
count,274.000000,274.000000
mean,56803.204380,9649.521898
std,13365.426169,2247.247932
min,29067.000000,4910.000000
25%,50312.000000,8597.500000
50%,56270.000000,9530.000000
75%,61447.000000,10441.500000
max,188514.000000,32545.000000


In [5]:
df.groupby("ticker")["transcript_word_count"].agg(["count", "mean", "min", "max"]).sort_values("mean")

,count,mean,min,max
ticker,,,,
AMZN,10,6289.700000,5151,7519
ORCL,11,6532.818182,4910,7946
GOOGL,10,7910.200000,5747,9182
NFLX,9,8161.666667,7395,9178
PTON,10,8327.200000,6782,9597
NVDA,8,8525.125000,8159,9000
AAPL,14,8599.000000,8091,9279
PYPL,8,8713.750000,7482,9655
UNH,9,8920.000000,7916,10068


In [6]:
suspect = df.loc[df["transcript_word_count"].idxmax()]
print(f"Ticker: {suspect['ticker']}")
print(f"Quarter: {suspect['quarter']}")
print(f"Date: {suspect['date_parsed']}")
print(f"Word count: {suspect['transcript_word_count']:,}")
print(f"\nFirst 500 chars:\n{suspect['transcript'][:500]}")
print(f"\nLast 500 chars:\n{suspect['transcript'][-500:]}")

Ticker: WMT
Quarter: 2020-Q4
Date: 2021-02-18 00:00:00
Word count: 32,545

First 500 chars:
Prepared Remarks:
Dan Binder -- Vice President, Investor Relations
Good morning and welcome to Walmart's 2021 Investment Community Meeting. Thank you all for joining us on the webcast. We appreciate your interest in Walmart, I know the executive team looks forward to sharing their strategies with you and answering your questions. Now, let me get a few of our usual statements out of the way.
The information presented at today's meeting should be viewed in conjunction with our press release and ea

Last 500 chars:
- Guggenheim -- Analyst
Stephanie Wissink -- Jefferies & Co. -- Analyst
Michael Lasser -- UBS -- Analyst
Robert F. Ohmes -- Bank of America Merrill Lynch -- Analyst
Kelly Bania -- BMO -- Analyst
Oliver Chen -- Cowen and Company -- Analyst
Ed Yruma -- KeyBanc Capital Markets -- Analyst
Greg Melich -- Evercore ISI -- Analyst
Chuck Grom -- Gordon Haskett -- Analyst
Chris Horvers -- JPMorgan

In [7]:
suspect_short = df.loc[df["transcript_word_count"].idxmin()]
print(f"Ticker: {suspect_short['ticker']}")
print(f"Quarter: {suspect_short['quarter']}")
print(f"Word count: {suspect_short['transcript_word_count']:,}")
print(f"\nFirst 500 chars:\n{suspect_short['transcript'][:500]}")

Ticker: ORCL
Quarter: 2021-Q4
Word count: 4,910

First 500 chars:
Prepared Remarks:
Operator
Welcome to Oracle's fourth-quarter 2021 earnings conference call. Now, I'd like to turn today's call over to Ken Bond, senior vice president. 
Ken Bond -- Senior Vice President
Thank you, Erica. Good afternoon, everyone, and welcome to Oracle's fourth-quarter and fiscal-year 2021 earnings conference call. A copy of the press release and financial tables, which includes a GAAP to non-GAAP reconciliation and other supplemental financial information, can be viewed and dow


In [8]:
# Other suspiciously long transcripts search

threshold = df["transcript_word_count"].quantile(0.95) # top 5%
print(f"95th percentile word count: {threshold:,.0f}")
print(f"Transcripts above 95th percentile: {(df['transcript_word_count'] > threshold).sum()}\n")

long_ones = df[df["transcript_word_count"] > threshold].sort_values("transcript_word_count", ascending = False)
print(long_ones[["ticker", "quarter", "date_parsed", "transcript_word_count"]].to_string())

95th percentile word count: 12,433
Transcripts above 95th percentile: 14

    ticker  quarter date_parsed  transcript_word_count
258    WMT  2020-Q4  2021-02-18                  32545
168    PFE  2021-Q4  2022-02-08                  17340
167    PFE  2021-Q3  2021-11-02                  14922
169    PFE  2022-Q2  2022-07-28                  14374
198   SBUX  2020-Q2  2020-04-28                  13893
164    PFE  2020-Q4  2021-02-02                  13767
166    PFE  2021-Q2  2021-07-28                  13603
171    PFE  2022-Q4  2023-01-31                  13403
265    XOM  2020-Q4  2021-02-02                  13305
39    BYND  2020-Q4  2021-02-25                  13099
30      BA  2019-Q2  2019-07-24                  12850
94     JNJ  2020-Q4  2021-01-26                  12727
206   SBUX  2022-Q4  2022-11-03                  12446
200   SBUX  2021-Q2  2021-04-27                  12440


In [9]:
for idx, row in long_ones.iterrows():
    print(f"\n{'='*60}")
    print(f"{row['ticker']} {row['quarter']} ({row['date_parsed'].date()}) - {row['transcript_word_count']:,} words")
    print(f"{'='*60}")
    print(row["transcript"][:300])


WMT 2020-Q4 (2021-02-18) - 32,545 words
Prepared Remarks:
Dan Binder -- Vice President, Investor Relations
Good morning and welcome to Walmart's 2021 Investment Community Meeting. Thank you all for joining us on the webcast. We appreciate your interest in Walmart, I know the executive team looks forward to sharing their strategies with yo

PFE 2021-Q4 (2022-02-08) - 17,340 words
Prepared Remarks:
Operator
Good day, everyone, and welcome to Pfizer's fourth quarter 2021 earnings conference call. Today's call is being recorded. At this time, I would like to turn the call over to Mr. Chris Stevo, senior vice president and chief investor relations officer.
Please go ahead, sir.


PFE 2021-Q3 (2021-11-02) - 14,922 words
Prepared Remarks:
Operator
Good day, everyone, and welcome to Pfizer's third quarter 2021 earnings conference call. Today's call is being recorded. At this time, I would like to turn the call over to Mr. Chris Stevo, senior vice president and chief investor relations officer

In [10]:
# drop walmart because earnings call mislabeled as investor day

before = len(df)
df = df[~((df["ticker"] == "WMT") & (df["quarter"] == "2020-Q4"))].copy()
after = len(df)
print(f"Dropped {before - after} row. Dataset: {before} -> {after} transcripts")

assert df["ticker"].nunique() == 30, "Lost a ticker"
assert len(df) == 273, f"Expected 273, got {len(df)}"
print("All 30 tickers still present.")


Dropped 1 row. Dataset: 274 -> 273 transcripts
All 30 tickers still present.


In [11]:
# Saving clean dataset as v2

processed_dir = REPO_ROOT / "data" / "processed"
output_path = processed_dir / "transcripts_v2.csv"

assert len(df) == 273, f"Refusing to save: expected 273 rows, got {len(df)}"
assert df["ticker"].nunique() == 30, "Refusing to save: not all 30 tickers present"

df.to_csv(output_path, index = False)

print(f"Saved: {output_path}")
print(f"Rows: {len(df)}, Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['date_parsed'].min().date()} -> {df['date_parsed'].max().date()}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

Saved: d:\Projects\risk-radar\data\processed\transcripts_v2.csv
Rows: 273, Tickers: 30
Date range: 2019-06-25 -> 2023-02-02
File size: 14.7 MB


In [12]:
# loading LM dataset

LM_PATH = REPO_ROOT / "data" / "raw" / "lexicons" / "Loughran-McDonald_MasterDictionary_1993-2025.csv"

assert LM_PATH.exists(), f"LM file not found at {LM_PATH}"

lm = pd.read_csv(LM_PATH)

print(f"Shape: {lm.shape}")
print(f"\nColumn names:")
for col in lm.columns:
    print(f" - {col}")
print(f"\nFirst row:")
print(lm.iloc[0])

Shape: (86553, 17)

Column names:
 - Word
 - Seq_num
 - Word Count
 - Word Proportion
 - Average Proportion
 - Std Dev
 - Doc Count
 - Negative
 - Positive
 - Uncertainty
 - Litigious
 - Strong_Modal
 - Weak_Modal
 - Constraining
 - Complexity
 - Syllables
 - Source

First row:
Word                   AARDVARK
Seq_num                       1
Word Count                  814
Word Proportion             0.0
Average Proportion          0.0
Std Dev                0.000004
Doc Count                   158
Negative                      0
Positive                      0
Uncertainty                   0
Litigious                     0
Strong_Modal                  0
Weak_Modal                    0
Constraining                  0
Complexity                    0
Syllables                     2
Source                12of12inf
Name: 0, dtype: object


In [14]:
# extracting the 4 LM word sets (imp variable)

print(f"Total LM dictionary size: {len(lm):,} words")

negative_words = set(lm.loc[lm["Negative"] > 0, "Word"].str.lower())
uncertainty_words = set(lm.loc[lm["Uncertainty"] > 0, "Word"].str.lower())
litigious_words = set(lm.loc[lm["Litigious"] > 0, "Word"].str.lower())
weak_modal_words = set(lm.loc[lm["Weak_Modal"] > 0, "Word"].str.lower())

print(f"\nCategory sizes:")
print(f" Negative: {len(negative_words):,} words")
print(f" Uncertainty: {len(uncertainty_words):,} words")
print(f" Litigious: {len(litigious_words):,} words")
print(f" Weak_Model: {len(weak_modal_words):,} words")

Total LM dictionary size: 86,553 words

Category sizes:
 Negative: 2,345 words
 Uncertainty: 297 words
 Litigious: 903 words
 Weak_Model: 27 words


In [22]:
# sanity check

print("Sample words from each category (random 10):\n")

import random
random.seed(42)

for name, word_set in [
    ("Negative", negative_words),
    ("Uncertainty", uncertainty_words),
    ("Litigious", litigious_words),
    ("Weak_Modal", weak_modal_words)
]:
    sample = random.sample(sorted(word_set), 10)
    print(f"{name}: {sample}")
    print()

Sample words from each category (random 10):

Negative: ['defamatory', 'annoyed', 'immature', 'fatality', 'erratically', 'detention', 'damaging', 'unsatisfactory', 'correcting', 'precludes']

Uncertainty: ['anticipation', 'anticipating', 'clarification', 'instabilities', 'nearly', 'unpredicted', 'anticipated', 'variants', 'improbability', 'vaguest']

Litigious: ['interposes', 'crimes', 'jurisprudence', 'perpetrating', 'disaffirmed', 'theretofor', 'wherewith', 'abrogations', 'statutory', 'thereon']

Weak_Modal: ['conceivable', 'somewhat', 'might', 'depends', 'depended', 'appears', 'could', 'suggest', 'appearing', 'appeared']



In [24]:
# coverage check

import re
from collections import Counter

# test transcript
sample_transcript = df.iloc[0]["transcript"]
sample_ticker = df.iloc[0]["ticker"]
sample_quarter = df.iloc[0]["quarter"]

print(f"Test transcript: {sample_ticker} {sample_quarter}")
print(f"Transcript length: {len(sample_transcript.split()):,} words\n")

tokens = re.findall(r"\b[a-z]+\b", sample_transcript.lower())
print(f"Tokens (alpha-only): {len(tokens):,}")

neg_hits = [t for t in tokens if t in negative_words]
unc_hits = [t for t in tokens if t in uncertainty_words]
lit_hits = [t for t in tokens if t in litigious_words]
wm_hits = [t for t in tokens if t in weak_modal_words]

print(f"\nMatches in this transcript:")
print(f" Negative: {len(neg_hits):,} matches ({len(set(neg_hits))} unique)")
print(f" Uncertainty: {len(unc_hits):,} matches ({len(set(unc_hits))} unique)")
print(f" Litigious: {len(lit_hits):,} matches ({len(set(lit_hits))} unique)")
print(f" Weak_Modal: {len(wm_hits):,} matches ({len(set(wm_hits))} unique)")

# what really matched
print(f"\nTop 10 negative words found:")
print(Counter(neg_hits).most_common(10))

print(f"\nTop 10 uncertainty words found:")
print(Counter(unc_hits).most_common(10))

Test transcript: AAPL 2019-Q3
Transcript length: 8,346 words

Tokens (alpha-only): 8,355

Matches in this transcript:
 Negative: 45 matches (15 unique)
 Uncertainty: 30 matches (14 unique)
 Litigious: 3 matches (3 unique)
 Weak_Modal: 22 matches (7 unique)

Top 10 negative words found:
[('question', 21), ('questions', 6), ('decline', 4), ('difficult', 2), ('negative', 2), ('limitation', 1), ('warned', 1), ('irregular', 1), ('loss', 1), ('suffering', 1)]

Top 10 uncertainty words found:
[('could', 6), ('almost', 5), ('may', 4), ('nearly', 3), ('probably', 2), ('maybe', 2), ('differ', 1), ('risk', 1), ('assumes', 1), ('believe', 1)]


In [ ]:
# auditing the top 30 most frequent matches per LM category across the full dataset

def tokenize(text: str) -> list[str]:
    """Lowercase, alpha-only word tokens."""
    return re.findall(r"\b[a-z]+\b", text.lower())

# Aggregating token counts across all 273 transcripts, per category
neg_counter = Counter()
unc_counter = Counter()
lit_counter = Counter()
wm_counter = Counter()

for transcript in df["transcript"]:
    tokens = tokenize(transcript)
    for tok in tokens:
        if tok in negative_words:
            neg_counter[tok] += 1
        if tok in uncertainty_words:
            unc_counter[tok] += 1
        if tok in litigious_words:
            lit_counter[tok] += 1
        if tok in weak_modal_words:
            wm_counter[tok] += 1

print(f"Audited {len(df)} transcripts.\n")

for name, counter in [
    ("NEGATIVE", neg_counter),
    ("UNCERTAINTY", unc_counter),
    ("LITIGIOUS", lit_counter),
    ("WEAK_MODAL", wm_counter),
]:
    total = sum(counter.values())
    top30 = counter.most_common(30)
    top30_total = sum(c for _, c in top30)
    print(f"=== {name} ===")
    print(f"Total matches across all transcripts: {total:,}")
    print(f"Unique matched words: {len(counter):,}")
    print(f"Top 30 account for: {top30_total:,} matches ({100*top30_total/total:.1f}% of total)")
    print(f"Top 30 most frequent:")
    for word, count in top30:
        pct = 100 * count / total
        print(f"  {word:25s} {count:6,}  ({pct:.1f}%)")
    print()

Audited 273 transcripts.

=== NEGATIVE ===
Total matches across all transcripts: 21,987
Unique matched words: 844
Top 30 account for: 14,822 matches (67.4% of total)
Top 30 most frequent:
  question                   7,347  (33.4%)
  questions                  1,691  (7.7%)
  challenges                   602  (2.7%)
  against                      518  (2.4%)
  decline                      476  (2.2%)
  negative                     423  (1.9%)
  difficult                    398  (1.8%)
  challenging                  323  (1.5%)
  declined                     276  (1.3%)
  loss                         275  (1.3%)
  challenge                    171  (0.8%)
  shortages                    169  (0.8%)
  disruptions                  158  (0.7%)
  recall                       156  (0.7%)
  negatively                   149  (0.7%)
  losses                       146  (0.7%)
  slower                       128  (0.6%)
  volatility                   128  (0.6%)
  problem                      125  (

In [ ]:
# applying stop lists to clean the LM word sets
NEGATIVE_STOPLIST = {"question", "questions"}
LITIGIOUS_STOPLIST = {"whatever", "whereas", "notwithstanding", "beneficial", "ratable"}

# Applying stop lists
negative_words_clean = negative_words - NEGATIVE_STOPLIST
litigious_words_clean = litigious_words - LITIGIOUS_STOPLIST
uncertainty_words_clean = uncertainty_words  # no changes
# weak_modal_words intentionally not used — dropped as a feature

print("Cleaned LM word sets:")
print(f"  Negative:    {len(negative_words):,} → {len(negative_words_clean):,} (removed {len(NEGATIVE_STOPLIST)})")
print(f"  Uncertainty: {len(uncertainty_words):,} → {len(uncertainty_words_clean):,} (no changes)")
print(f"  Litigious:   {len(litigious_words):,} → {len(litigious_words_clean):,} (removed {len(LITIGIOUS_STOPLIST)})")
print(f"  Weak_Modal:  dropped as feature (90%+ overlap with Uncertainty)")

# Verify stop-list words are actually gone
assert "question" not in negative_words_clean, "question still in negative_words_clean"
assert "questions" not in negative_words_clean, "questions still in negative_words_clean"
assert "notwithstanding" not in litigious_words_clean, "notwithstanding still in litigious_words_clean"

print("\nStop-list assertions passed.")

Cleaned LM word sets:
  Negative:    2,345 → 2,343 (removed 2)
  Uncertainty: 297 → 297 (no changes)
  Litigious:   903 → 898 (removed 5)
  Weak_Modal:  dropped as feature (90%+ overlap with Uncertainty)

Stop-list assertions passed.


In [ ]:
import textstat

# Forward-looking word list
FORWARD_LOOKING_WORDS = {
    "expect", "expects", "expected", "expecting",
    "anticipate", "anticipates", "anticipated", "anticipating",
    "guidance", "outlook", "forecast", "forecasts", "forecasting",
    "project", "projects", "projecting", "projected",
    "plan", "plans", "planning", "planned",
    "target", "targets", "targeting", "targeted",
    "foresee", "foresees", "foreseen",
    "intend", "intends", "intending", "intended",
    "will",
}

# Regex for numeric tokens (matches integers, decimals, percentages, dollar amounts)
NUMERIC_RE = re.compile(r"\b\d[\d,.]*\b|\$\d[\d,.]*|%")

def extract_features(transcript: str) -> dict:
    """
    Extract 9 features from a single transcript.
    Returns a dict with feature_name -> value.
    """

    tokens = tokenize(transcript)
    word_count = len(tokens)

    # Sentence-level features (textstat handles its own tokenization)
    sentence_count = textstat.sentence_count(transcript)
    fk_grade = textstat.flesch_kincaid_grade(transcript)

    # Avoiding division by zero
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0.0

    # Numeric density: counting numeric matches in original (mixed-case) text
    numeric_matches = NUMERIC_RE.findall(transcript)
    numeric_density = (len(numeric_matches) / word_count) * 1000 if word_count > 0 else 0.0

    # Lexicon-based counts
    negative_count = sum(1 for tok in tokens if tok in negative_words_clean)
    uncertainty_count = sum(1 for tok in tokens if tok in uncertainty_words_clean)
    litigious_count = sum(1 for tok in tokens if tok in litigious_words_clean)
    forward_looking_count = sum(1 for tok in tokens if tok in FORWARD_LOOKING_WORDS)

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "flesch_kincaid_grade": fk_grade,
        "numeric_density": numeric_density,
        "negative_count": negative_count,
        "uncertainty_count": uncertainty_count,
        "litigious_count": litigious_count,
        "forward_looking_count": forward_looking_count
    }

# Sanity check
sample_features = extract_features(df.iloc[0]["transcript"])
print(f"Feature extraction on {df.iloc[0]['ticker']} {df.iloc[0]['quarter']}:")
for k, v in sample_features.items():
    print(f" {k:25s} {v:10.2f}" if isinstance(v, float) else f" {k:25s} {v:10d}")

Feature extraction on AAPL 2019-Q3:
 word_count                      8355
 sentence_count                   440
 avg_sentence_length            18.99
 flesch_kincaid_grade           10.24
 numeric_density                26.21
 negative_count                    18
 uncertainty_count                 30
 litigious_count                    3
 forward_looking_count             67


In [28]:
# applying extract_features to all 273 transcripts

import time

print("Extracting features from 273 transcripts...")
start = time.time()

# Applying extract_features to every transcript
# .apply() with a function that returns a dict produces a Series of dicts.
# pd.json_normalize turns those dicts into proper DataFrame columns.
feature_dicts = df["transcript"].apply(extract_features)
features_df = pd.json_normalize(feature_dicts)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f} seconds.\n")

# Combining identifier columns with feature columns
identifier_cols = df[["ticker", "quarter", "date_parsed"]].reset_index(drop = True)
features_df = features_df.reset_index(drop = True)

features_full = pd.concat([identifier_cols, features_df], axis = 1)

print(f"Shape: {features_full.shape}")
print(f"Columns: {list(features_full.columns)}")
print(f"\nFirst 3 rows:")
print(features_full.head(3))

Extracting features from 273 transcripts...
Done in 5.9 seconds.

Shape: (273, 12)
Columns: ['ticker', 'quarter', 'date_parsed', 'word_count', 'sentence_count', 'avg_sentence_length', 'flesch_kincaid_grade', 'numeric_density', 'negative_count', 'uncertainty_count', 'litigious_count', 'forward_looking_count']

First 3 rows:
  ticker  quarter date_parsed  word_count  sentence_count  \
0   AAPL  2019-Q3  2019-07-30        8355             440   
1   AAPL  2020-Q1  2020-01-28        8267             459   
2   AAPL  2020-Q2  2020-04-30        8278             442   

   avg_sentence_length  flesch_kincaid_grade  numeric_density  negative_count  \
0            18.988636             10.239787        26.211849              18   
1            18.010893             10.009808        24.434499              27   
2            18.728507             10.225606        16.791496              51   

   uncertainty_count  litigious_count  forward_looking_count  
0                 30                3     

In [ ]:
# sanity-check feature distributions across all 273 transcripts
print("=" * 70)
print("FEATURE DISTRIBUTIONS — all 273 transcripts")
print("=" * 70)

feature_cols = [
    "word_count", "sentence_count", "avg_sentence_length",
    "flesch_kincaid_grade", "numeric_density",
    "negative_count", "uncertainty_count", "litigious_count",
    "forward_looking_count",
]

# describe() for all features
print(features_full[feature_cols].describe().round(2).to_string())

# Looking for any null values (should be zero)
print(f"\n{'-' * 70}")
print("Null counts per feature (expecting all zeros):")
print(features_full[feature_cols].isnull().sum().to_string())

# Looking for any zero values that might indicate failed extraction
print(f"\n{'-' * 70}")
print("Zero counts per feature:")
zero_counts = (features_full[feature_cols] == 0).sum()
print(zero_counts.to_string())

# min/max examples for sanity
print(f"\n{'-' * 70}")
print("Extreme rows for negative_count:")
neg_min_idx = features_full["negative_count"].idxmin()
neg_max_idx = features_full["negative_count"].idxmax()
print(f"  MIN negative_count: {features_full.loc[neg_min_idx, 'ticker']} {features_full.loc[neg_min_idx, 'quarter']} → {features_full.loc[neg_min_idx, 'negative_count']}")
print(f"  MAX negative_count: {features_full.loc[neg_max_idx, 'ticker']} {features_full.loc[neg_max_idx, 'quarter']} → {features_full.loc[neg_max_idx, 'negative_count']}")

print(f"\nExtreme rows for litigious_count:")
lit_max_idx = features_full["litigious_count"].idxmax()
print(f"  MAX litigious_count: {features_full.loc[lit_max_idx, 'ticker']} {features_full.loc[lit_max_idx, 'quarter']} → {features_full.loc[lit_max_idx, 'litigious_count']}")

FEATURE DISTRIBUTIONS — all 273 transcripts
       word_count  sentence_count  avg_sentence_length  flesch_kincaid_grade  numeric_density  negative_count  uncertainty_count  litigious_count  forward_looking_count
count      273.00          273.00               273.00                273.00           273.00          273.00             273.00           273.00                 273.00
mean      9590.59          542.85                17.81                  9.86            20.78           47.43              72.23             6.70                  78.40
std       1767.14          105.02                 2.00                  1.09             9.13           21.60              24.71             6.87                  37.26
min       4844.00          282.00                13.10                  7.59             3.55           11.00              25.00             0.00                  17.00
25%       8603.00          487.00                16.51                  9.15            13.83           31.00  

In [ ]:
# computing per-ticker z-scores for all 9 raw features

raw_feature_cols = [
    "word_count", "sentence_count", "avg_sentence_length",
    "flesch_kincaid_grade", "numeric_density",
    "negative_count", "uncertainty_count", "litigious_count",
    "forward_looking_count",
]

# Computing z-scores within each ticker group
# .transform() applies a function within each group and returns a same-shape result
for col in raw_feature_cols:
    grouped = features_full.groupby("ticker")[col]
    mean = grouped.transform("mean")
    std = grouped.transform("std")
    
    # Avoiding division by zero: if std is 0 (all identical values for a ticker),
    # seting z-score to 0 instead of NaN
    z_col = (features_full[col] - mean) / std
    z_col = z_col.fillna(0.0)  # handles std=0 edge case
    
    features_full[f"{col}_zscore"] = z_col

# Verifying shape
print(f"Shape after z-scoring: {features_full.shape}")
print(f"\nAll columns:")
for c in features_full.columns:
    print(f"  {c}")

# Sanity check: z-scores within each ticker should have mean ≈ 0 and std ≈ 1
print(f"\n{'-' * 70}")
print("Verification: within-ticker mean and std for each z-scored feature")
print("(Mean should be ~0, std should be ~1 for tickers with enough samples)")
print(f"{'-' * 70}")

zscore_cols = [f"{c}_zscore" for c in raw_feature_cols]
within_ticker = features_full.groupby("ticker")[zscore_cols].agg(["mean", "std"])
print(within_ticker.iloc[:3].round(3).to_string())  # first 3 tickers
print(f"\n(Showing first 3 tickers; full result has {features_full['ticker'].nunique()} tickers)")

Shape after z-scoring: (273, 21)

All columns:
  ticker
  quarter
  date_parsed
  word_count
  sentence_count
  avg_sentence_length
  flesch_kincaid_grade
  numeric_density
  negative_count
  uncertainty_count
  litigious_count
  forward_looking_count
  word_count_zscore
  sentence_count_zscore
  avg_sentence_length_zscore
  flesch_kincaid_grade_zscore
  numeric_density_zscore
  negative_count_zscore
  uncertainty_count_zscore
  litigious_count_zscore
  forward_looking_count_zscore

----------------------------------------------------------------------
Verification: within-ticker mean and std for each z-scored feature
(Mean should be ~0, std should be ~1 for tickers with enough samples)
----------------------------------------------------------------------
       word_count_zscore      sentence_count_zscore      avg_sentence_length_zscore      flesch_kincaid_grade_zscore      numeric_density_zscore      negative_count_zscore      uncertainty_count_zscore      litigious_count_zscore    

In [31]:
# saving the feature-engineered dataset

processed_dir = REPO_ROOT / "data" / "processed"
output_path = processed_dir / "features.csv"

# Pre-save asserts
assert features_full.shape == (273, 21), f"Expected (273, 21), got {features_full.shape}"
assert features_full["ticker"].nunique() == 30, "Lost a ticker"
assert features_full.isnull().sum().sum() == 0, "NaN values present in features"

# Verifying all 9 z-score columns exist
expected_zscore_cols = {f"{c}_zscore" for c in raw_feature_cols}
actual_zscore_cols = {c for c in features_full.columns if c.endswith("_zscore")}
assert expected_zscore_cols == actual_zscore_cols, (
    f"Z-score column mismatch.\n"
    f"Missing: {expected_zscore_cols - actual_zscore_cols}\n"
    f"Extra: {actual_zscore_cols - expected_zscore_cols}"
)

# Save
features_full.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {features_full.shape}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"Tickers: {features_full['ticker'].nunique()}")
print(f"Date range: {features_full['date_parsed'].min().date()} → {features_full['date_parsed'].max().date()}")

Saved: d:\Projects\risk-radar\data\processed\features.csv
Shape: (273, 21)
File size: 74.3 KB
Tickers: 30
Date range: 2019-06-25 → 2023-02-02


In [34]:
# loading price data: 30 tickers + SPY into one long format df

import yfinance as yf

# pulling SPY
print("Downloading SPY...")
spy_raw = yf.download(
    tickers = "SPY",
    start = "2017-01-01", 
    end = "2024-12-31",
    interval = "1d",
    auto_adjust = True,
    progress = False
)

# Flatten MultiIndex — drop the ticker level, keep only 'Open'/'High'/etc
if isinstance(spy_raw.columns, pd.MultiIndex):
    spy_raw.columns = spy_raw.columns.droplevel(1)
    print("Flattened MultiIndex columns")

print(f"Final shape: {spy_raw.shape}")
print(f"Final columns: {list(spy_raw.columns)}")
print(f"Index name: {spy_raw.index.name}")
print(f"\nFirst 3 rows:")
print(spy_raw.head(3))

# saving SPY 
spy_path = REPO_ROOT / "data" / "raw" / "prices" / "SPY.csv"
spy_raw.to_csv(spy_path)
print(f"Saved SPY to {spy_path}")

Flattened MultiIndex columns
Final shape: (2011, 5)
Final columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Index name: Date

First 3 rows:
Price            Close        High         Low        Open    Volume
Date                                                                
2017-01-03  194.467850  194.977241  193.293652  194.295163  91366500
2017-01-04  195.624786  195.771560  194.787306  194.795935  78744400
2017-01-05  195.469360  195.624776  194.675052  195.357130  78379000
Saved SPY to d:\Projects\risk-radar\data\raw\prices\SPY.csv


In [ ]:
# Verifying the above file
with open(spy_path, "r") as f:
    for i, line in enumerate(f):
        if i < 5:
            print(f"Line {i}: {line.rstrip()}")
        else:
            break

print()
spy_loaded = pd.read_csv(spy_path, index_col="Date", parse_dates=True)
print(f"Shape on reload: {spy_loaded.shape}")
print(f"Columns on reload: {list(spy_loaded.columns)}")
print(f"Index dtype: {spy_loaded.index.dtype}")
print(f"\nFirst 3 rows:")
print(spy_loaded.head(3))

Line 0: Date,Close,High,Low,Open,Volume
Line 1: 2017-01-03,194.4678497314453,194.97724112010118,193.29365159666258,194.2951630733348,91366500
Line 2: 2017-01-04,195.62478637695312,195.77155950439442,194.78730610485513,194.79593516853652,78744400
Line 3: 2017-01-05,195.4693603515625,195.62477570835244,194.67505178947582,195.3571298332533,78379000
Line 4: 2017-01-06,196.16873168945312,196.63495146380882,195.03769192320487,195.58162597173967,71559900

Shape on reload: (2011, 5)
Columns on reload: ['Close', 'High', 'Low', 'Open', 'Volume']
Index dtype: datetime64[us]

First 3 rows:
                 Close        High         Low        Open    Volume
Date                                                                
2017-01-03  194.467850  194.977241  193.293652  194.295163  91366500
2017-01-04  195.624786  195.771560  194.787306  194.795935  78744400
2017-01-05  195.469360  195.624776  194.675052  195.357130  78379000


In [36]:
# computing excess returns for 3 windows
PRICES_DIR = REPO_ROOT / "data" / "raw" / "prices"
WINDOWS = [1, 3, 5]

# Loading SPY once
spy = pd.read_csv(PRICES_DIR / "SPY.csv", index_col="Date", parse_dates=True)
spy_close = spy["Close"]
print(f"SPY loaded: {len(spy_close)} trading days, {spy_close.index.min().date()} → {spy_close.index.max().date()}")

# Cache ticker prices so we don't re-read 30 files repeatedly
ticker_close_cache = {}
def get_ticker_close(ticker: str) -> pd.Series:
    if ticker not in ticker_close_cache:
        path = PRICES_DIR / f"{ticker}.csv"
        df_t = pd.read_csv(path, index_col="Date", parse_dates=True)
        ticker_close_cache[ticker] = df_t["Close"]
    return ticker_close_cache[ticker]


def compute_excess_returns(ticker: str, call_date: pd.Timestamp, windows: list[int]) -> dict:
    """
    For one transcript, compute excess returns at each window.
    Returns a dict like {'excess_return_1d': 0.012, 'excess_return_3d': -0.005, ...}
    """
    stock_close = get_ticker_close(ticker)
    
    # Find the call date in the price index — or the next available trading day
    # searchsorted returns the position where call_date *would* be inserted; if call_date
    # is itself a trading day, this gives us its position.
    pos = stock_close.index.searchsorted(call_date)
    
    # Edge case: call_date is after the last available price date
    if pos >= len(stock_close):
        return {f"excess_return_{w}d": None for w in windows}
    
    # Edge case: call_date is before the first price date (shouldn't happen but guard)
    if pos == 0 and stock_close.index[0] > call_date:
        return {f"excess_return_{w}d": None for w in windows}
    
    # Day 0 = first trading day at or after the call
    day_0_date = stock_close.index[pos]
    day_0_stock = stock_close.iloc[pos]
    
    # Find SPY's day 0 the same way (could be a different position than stock's pos
    # if a stock had a trading halt — rare, but defensive)
    spy_pos = spy_close.index.searchsorted(day_0_date)
    if spy_pos >= len(spy_close):
        return {f"excess_return_{w}d": None for w in windows}
    day_0_spy = spy_close.iloc[spy_pos]
    
    results = {}
    for w in windows:
        stock_target_pos = pos + w
        spy_target_pos = spy_pos + w
        
        # Edge case: not enough forward data
        if stock_target_pos >= len(stock_close) or spy_target_pos >= len(spy_close):
            results[f"excess_return_{w}d"] = None
            continue
        
        stock_return = (stock_close.iloc[stock_target_pos] - day_0_stock) / day_0_stock
        spy_return = (spy_close.iloc[spy_target_pos] - day_0_spy) / day_0_spy
        excess = stock_return - spy_return
        results[f"excess_return_{w}d"] = excess
    
    return results


# Apply to every transcript
print(f"\nComputing excess returns for {len(features_full)} transcripts × {len(WINDOWS)} windows...")
returns_records = []
for _, row in features_full.iterrows():
    ticker = row["ticker"]
    call_date = pd.Timestamp(row["date_parsed"])
    returns = compute_excess_returns(ticker, call_date, WINDOWS)
    returns["ticker"] = ticker
    returns["quarter"] = row["quarter"]
    returns["date_parsed"] = call_date
    returns_records.append(returns)

returns_df = pd.DataFrame(returns_records)
print(f"Shape: {returns_df.shape}")
print(f"\nFirst 5 rows:")
print(returns_df.head())
print(f"\nNull counts (transcripts where some window couldn't be computed):")
print(returns_df[[f"excess_return_{w}d" for w in WINDOWS]].isnull().sum())

SPY loaded: 2011 trading days, 2017-01-03 → 2024-12-30

Computing excess returns for 273 transcripts × 3 windows...
Shape: (273, 6)

First 5 rows:
   excess_return_1d  excess_return_3d  excess_return_5d ticker  quarter  \
0          0.031345          0.004137         -0.013460   AAPL  2019-Q3   
1          0.021758         -0.009963         -0.002987   AAPL  2020-Q1   
2          0.010374          0.027566          0.043472   AAPL  2020-Q2   
3          0.096787          0.121258          0.152131   AAPL  2020-Q3   
4         -0.045593         -0.060651         -0.029227   AAPL  2020-Q4   

  date_parsed  
0  2019-07-30  
1  2020-01-28  
2  2020-04-30  
3  2020-07-30  
4  2020-10-29  

Null counts (transcripts where some window couldn't be computed):
excess_return_1d    0
excess_return_3d    0
excess_return_5d    0
dtype: int64


In [ ]:
# applying median splits to create binary labels, save labels.csv

WINDOWS = [1, 3, 5]

# Reordering columns
returns_df = returns_df[["ticker", "quarter", "date_parsed"] + [f"excess_return_{w}d" for w in WINDOWS]]

# Computing median for each window's returns
medians = {w: returns_df[f"excess_return_{w}d"].median() for w in WINDOWS}
print("Median excess returns:")
for w, m in medians.items():
    print(f"  {w}-day window: {m:+.4f} ({m*100:+.2f}%)")

# Applying median split to create binary labels
# y = 1 if excess return is BELOW median (high-risk outcome)
# y = 0 if excess return is AT or ABOVE median (low-risk outcome)
for w in WINDOWS:
    returns_df[f"y_{w}d"] = (returns_df[f"excess_return_{w}d"] < medians[w]).astype(int)

# Verifying class balance — should be very close to 50/50 by construction
print("\nClass balance per window:")
for w in WINDOWS:
    pos_count = returns_df[f"y_{w}d"].sum()
    total = len(returns_df)
    print(f"  y_{w}d:  {pos_count} positive ({100*pos_count/total:.1f}%) | {total-pos_count} negative ({100*(total-pos_count)/total:.1f}%)")

# Save
labels_path = REPO_ROOT / "data" / "processed" / "labels.csv"

# Pre-save asserts
assert returns_df.shape == (273, 9), f"Expected (273, 9), got {returns_df.shape}"
assert returns_df["ticker"].nunique() == 30, "Lost a ticker"
for w in WINDOWS:
    assert returns_df[f"y_{w}d"].isin([0, 1]).all(), f"Non-binary values in y_{w}d"
    pos_pct = returns_df[f"y_{w}d"].sum() / len(returns_df)
    assert 0.45 < pos_pct < 0.55, f"y_{w}d class imbalance: {pos_pct:.3f}"

returns_df.to_csv(labels_path, index=False)

print(f"\nSaved: {labels_path}")
print(f"Shape: {returns_df.shape}")
print(f"File size: {labels_path.stat().st_size / 1024:.1f} KB")
print(f"\nFirst 3 rows:")
print(returns_df.head(3))

Median excess returns:
  1-day window: -0.0024 (-0.24%)
  3-day window: -0.0064 (-0.64%)
  5-day window: -0.0056 (-0.56%)

Class balance per window:
  y_1d:  136 positive (49.8%) | 137 negative (50.2%)
  y_3d:  136 positive (49.8%) | 137 negative (50.2%)
  y_5d:  136 positive (49.8%) | 137 negative (50.2%)

Saved: d:\Projects\risk-radar\data\processed\labels.csv
Shape: (273, 9)
File size: 24.9 KB

First 3 rows:
  ticker  quarter date_parsed  excess_return_1d  excess_return_3d  \
0   AAPL  2019-Q3  2019-07-30          0.031345          0.004137   
1   AAPL  2020-Q1  2020-01-28          0.021758         -0.009963   
2   AAPL  2020-Q2  2020-04-30          0.010374          0.027566   

   excess_return_5d  y_1d  y_3d  y_5d  
0         -0.013460     0     0     1  
1         -0.002987     0     1     0  
2          0.043472     0     0     0  


In [ ]:
# joining features.csv and labels.csv into a single modeling dataset

features_path = REPO_ROOT / "data" / "processed" / "features.csv"
labels_path = REPO_ROOT / "data" / "processed" / "labels.csv"
dataset_path = REPO_ROOT / "data" / "processed" / "dataset.csv"

# Loading both files fresh 
features = pd.read_csv(features_path, parse_dates=["date_parsed"])
labels = pd.read_csv(labels_path, parse_dates=["date_parsed"])

print(f"Features: {features.shape}")
print(f"Labels:   {labels.shape}")

# Merging on the three identifier columns
# how="inner" means: only keep rows that exist in BOTH files
# This is what we want — every transcript should have features AND labels
dataset = features.merge(
    labels,
    on=["ticker", "quarter", "date_parsed"],
    how="inner",
    validate="one_to_one",  # raise if any duplicate keys exist
)

print(f"\nDataset after merge: {dataset.shape}")

# Pre-save asserts
assert dataset.shape[0] == 273, f"Expected 273 rows, got {dataset.shape[0]}"
assert dataset.shape[1] == 27, f"Expected 27 columns, got {dataset.shape[1]}"  # 3 ids + 18 features + 3 returns + 3 labels
assert dataset["ticker"].nunique() == 30, "Lost a ticker"
assert dataset.isnull().sum().sum() == 0, "NaN values present"
for w in [1, 3, 5]:
    assert dataset[f"y_{w}d"].isin([0, 1]).all(), f"y_{w}d has non-binary values"

dataset.to_csv(dataset_path, index=False)

print(f"\nSaved: {dataset_path}")
print(f"File size: {dataset_path.stat().st_size / 1024:.1f} KB")
print(f"\nColumn list ({len(dataset.columns)} columns):")
for c in dataset.columns:
    print(f"  {c}")

print(f"\nFirst 2 rows (transposed for readability):")
print(dataset.head(2).T)

Features: (273, 21)
Labels:   (273, 9)

Dataset after merge: (273, 27)

Saved: d:\Projects\risk-radar\data\processed\dataset.csv
File size: 90.7 KB

Column list (27 columns):
  ticker
  quarter
  date_parsed
  word_count
  sentence_count
  avg_sentence_length
  flesch_kincaid_grade
  numeric_density
  negative_count
  uncertainty_count
  litigious_count
  forward_looking_count
  word_count_zscore
  sentence_count_zscore
  avg_sentence_length_zscore
  flesch_kincaid_grade_zscore
  numeric_density_zscore
  negative_count_zscore
  uncertainty_count_zscore
  litigious_count_zscore
  forward_looking_count_zscore
  excess_return_1d
  excess_return_3d
  excess_return_5d
  y_1d
  y_3d
  y_5d

First 2 rows (transposed for readability):
                                                0                    1
ticker                                       AAPL                 AAPL
quarter                                   2019-Q3              2020-Q1
date_parsed                   2019-07-30 00:00:00 